# Агрегации, JOIN и окна — 30 заданий

Практика на eBay; решений нет.

## Результаты обучения

После **JOIN, агрегации и окна** вы должны объяснить transformation как plan, предсказать action/jobs/shuffle, связать schema/grain с результатом и доказать физическую эффективность через explain/UI/metrics.

## Ментальная модель исполнения

GroupBy и крупный JOIN обычно создают shuffle. Broadcast переносит малую сторону. Window сохраняет grain, но требует partition/order/frame и часто сортировки.

```text
transformations → unresolved logical plan
       ↓ analysis (catalog/types)
   optimized logical plan (Catalyst)
       ↓ physical planning / AQE
job → stage → shuffle → stage
       tasks             tasks
       └──── executors ─────┘
```
Action создаёт job. Один notebook/application может породить много jobs, а один job —
несколько stages. `repartition`, join и groupBy часто добавляют Exchange.

## Данные eBay

Grain eBay — `itemid` в `snapshot_dt`; 2 501 511 строк, 24 колонки, Parquet/Snappy.
Partition column — дата снимка. Цена, продавец, категории и доставка денормализованы.
Перед `latest item` или dedup проверяйте уникальность пары и задавайте tie-breaker.

Полная схема и проверки качества находятся в `data-catalog`. Raw read-only, результаты — в личном `spark_training`.

## Алгоритм решения

1. Зафиксируйте входной и целевой grain. 2. Выберите только нужные columns/rows. 3. Соберите transformation без action. 4. Проверьте schema и explain. 5. Предскажите partitions/shuffle. 6. Выполните минимальный action/write. 7. Повторно прочитайте и сверяйте keys/metrics. 8. Сохраните evidence.

Докажите cardinality до JOIN, используйте предагрегацию и сверяйте ключи/суммы после.

## Типичные ошибки

- Вызывать count/show после каждого шага и создавать лишние jobs.
- Использовать Python UDF при наличии встроенной функции.
- Делать repartition без понимания Exchange и целевого файла.
- Broadcast большой стороны или collect на driver.
- Кэшировать одноразовый DataFrame без materialization/unpersist.
- Измерять скорость при разных результатах или непрогретом JVM.

## Самопроверка

1. Какой action создаёт job? 2. Где появится shuffle? 3. Сколько input/output partitions? 4. Видит ли Catalyst выражение? 5. Каков grain после JOIN/window? 6. Как проверить idempotent rerun?

## Подробная теория

### 1. Grain

Перед JOIN назовите grain и кратность. N:M размножает строки и искажает суммы.

### 2. Aggregation

Предагрегируйте каждую деталь до ключа соединения.

### 3. Join

Broadcast малой стороны избегает shuffle; крупные стороны обычно требуют exchange и sort/hash.

### 4. Windows

Partition задаёт группу, order — последовательность, frame — диапазон вокруг строки.

### 5. Контроль

Сверяйте count, distinct key и суммы до и после JOIN.

## Сдача

Каждое задание записывает непустой Parquet в личный HDFS и evidence с transformation, observation и explanation. Checker использует активную SparkSession.

In [ ]:
import os,sys
sys.path.insert(0,'/opt/lab/spark-training')
from check_task import check_task,save_evidence
from pyspark.sql import SparkSession,functions as F,types as T,Window
spark=SparkSession.builder.appName('spark-training').enableHiveSupport().getOrCreate()
USER=os.environ.get('HDFS_USER',os.environ.get('HADOOP_USER_NAME','student'))
ROOT=f'hdfs://namenode:8020/user/{USER}/spark_training'
SOURCE='hdfs://namenode:8020/data/raw/ebay'
ebay=spark.read.parquet(SOURCE)
print('Spark',spark.version,'rows',ebay.count(),'columns',len(ebay.columns))

### Задание 1. count variants

Создайте результат по теме **count variants** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_01")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',1)

### Задание 2. sum avg

Создайте результат по теме **sum avg** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_02")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',2)

### Задание 3. min max

Создайте результат по теме **min max** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_03")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',3)

### Задание 4. approx distinct

Создайте результат по теме **approx distinct** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_04")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',4)

### Задание 5. groupBy

Создайте результат по теме **groupBy** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_05")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',5)

### Задание 6. pivot

Создайте результат по теме **pivot** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_06")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',6)

### Задание 7. rollup

Создайте результат по теме **rollup** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_07")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',7)

### Задание 8. cube

Создайте результат по теме **cube** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_08")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',8)

### Задание 9. inner join

Создайте результат по теме **inner join** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_09")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',9)

### Задание 10. left join

Создайте результат по теме **left join** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_10")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',10)

### Задание 11. left semi

Создайте результат по теме **left semi** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_11")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',11)

### Задание 12. left anti

Создайте результат по теме **left anti** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_12")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',12)

### Задание 13. null-safe join

Создайте результат по теме **null-safe join** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_13")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',13)

### Задание 14. join cardinality

Создайте результат по теме **join cardinality** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_14")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',14)

### Задание 15. pre-aggregation

Создайте результат по теме **pre-aggregation** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_15")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',15)

### Задание 16. broadcast join

Создайте результат по теме **broadcast join** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_16")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',16)

### Задание 17. self join

Создайте результат по теме **self join** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_17")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',17)

### Задание 18. row_number

Создайте результат по теме **row_number** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_18")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',18)

### Задание 19. rank dense_rank

Создайте результат по теме **rank dense_rank** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_19")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',19)

### Задание 20. lag lead

Создайте результат по теме **lag lead** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_20")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',20)

### Задание 21. running sum

Создайте результат по теме **running sum** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_21")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',21)

### Задание 22. moving average

Создайте результат по теме **moving average** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_22")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',22)

### Задание 23. top N group

Создайте результат по теме **top N group** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_23")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',23)

### Задание 24. first last

Создайте результат по теме **first last** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_24")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',24)

### Задание 25. percentile

Создайте результат по теме **percentile** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_25")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',25)

### Задание 26. gaps islands

Создайте результат по теме **gaps islands** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_26")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',26)

### Задание 27. sessions

Создайте результат по теме **sessions** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_27")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',27)

### Задание 28. cohort

Создайте результат по теме **cohort** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_28")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',28)

### Задание 29. retention

Создайте результат по теме **retention** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_29")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',29)

### Задание 30. analytics mart

Создайте результат по теме **analytics mart** и запишите `mode("overwrite").parquet(f"{ROOT}/analytics/task_30")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'analytics',30)